# Saheli — GGUF Conversion (Colab)

Pulls the LoRA adapter from HuggingFace, merges it into base Gemma 4 E4B, converts to GGUF F16, and quantises to Q4_K_M for llama.cpp offline inference.

**Runtime:** T4 GPU is enough. The whole flow takes ~20 minutes end-to-end.

**Disk needed:** ~35 GB (Colab gives ~80 GB — fine).

## Step 1 — Configure
Edit the constants below to match your HF repo.

In [ ]:
HF_USER       = 'sriramarivazhagan'            # ← your HF username
ADAPTER_REPO  = f'{HF_USER}/saheli-gemma4-e4b' # where you pushed the adapter
BASE_MODEL    = 'unsloth/gemma-4-E4B-it'        # same base used for fine-tuning

GGUF_BASENAME = 'saheli-gemma4-e4b'             # output filename stem
QUANT         = 'Q4_K_M'                        # Q4_K_M = best quality/size for 4B

# Push the final GGUF back to the SAME HF repo (keeps adapter + GGUF together)
PUSH_GGUF_TO_HF = True

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

print('Adapter repo :', ADAPTER_REPO)
print('Base model   :', BASE_MODEL)
print('Output file  :', f'{GGUF_BASENAME}-{QUANT.lower()}.gguf')

## Step 2 — Install dependencies

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps unsloth
!pip install -q huggingface_hub hf_transfer sentencepiece

import torch, unsloth
print('torch   :', torch.__version__)
print('unsloth :', unsloth.__version__)
print('CUDA    :', torch.cuda.is_available())

## Step 3 — Authenticate with HuggingFace

Colab left sidebar → 🔑 → add `HF_TOKEN` secret with read+write access.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('Logged in to HuggingFace.')

## Step 4 — Download base Gemma 4 E4B + adapter

In [ ]:
from huggingface_hub import snapshot_download

BASE_DIR    = snapshot_download(BASE_MODEL,   local_dir='/content/base')
ADAPTER_DIR = snapshot_download(ADAPTER_REPO, local_dir='/content/adapter')

print('Base dir    :', BASE_DIR)
print('Adapter dir :', ADAPTER_DIR)

!df -h /content

## Step 5 — Merge LoRA adapter into base model (fp16)

Uses PEFT's `merge_and_unload` to produce a standalone merged model that llama.cpp can convert.

In [ ]:
from unsloth import FastModel
import os

GGUF_OUT_DIR = '/content/gguf_out'
os.makedirs(GGUF_OUT_DIR, exist_ok=True)

print('Loading base + LoRA adapter via Unsloth (4-bit, short context for VRAM)...')
model, tokenizer = FastModel.from_pretrained(
    model_name     = ADAPTER_DIR,
    max_seq_length = 512,          # minimal — only needed for export, not inference
    load_in_4bit   = True,
)

print('Exporting merged GGUF Q4_K_M...')
model.save_pretrained_gguf(
    GGUF_OUT_DIR,
    tokenizer,
    quantization_method = 'q4_k_m',
)

for f in os.listdir(GGUF_OUT_DIR):
    size = os.path.getsize(f'{GGUF_OUT_DIR}/{f}') / 1e9
    print(f'  {f}  ({size:.2f} GB)')

!df -h /content

## Step 6 — Find the GGUF file

Unsloth writes the GGUF directly in the previous step. Steps 6–8 (llama.cpp build, F16 convert, quantise) are no longer needed.

In [ ]:
import os, glob

# Unsloth names the file with the quant suffix, e.g. model-Q4_K_M.gguf
matches = glob.glob('/content/gguf_out/*.gguf')
assert matches, 'No GGUF found — Step 5 may have failed'
Q_GGUF = matches[0]
print('GGUF file:', Q_GGUF)
print(f'Size     : {os.path.getsize(Q_GGUF)/1e9:.2f} GB')

In [ ]:
F16_GGUF = f'/content/{GGUF_BASENAME}-f16.gguf'

!python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/merged \
    --outfile {F16_GGUF} \
    --outtype f16

import os
print(f'\nF16 GGUF: {F16_GGUF}  ({os.path.getsize(F16_GGUF)/1e9:.2f} GB)')

In [ ]:
import os, subprocess

Q_GGUF = f'/content/{GGUF_BASENAME}-{QUANT.lower()}.gguf'

# Binary location differs between llama.cpp CMake versions
candidates = [
    '/content/llama.cpp/build/bin/llama-quantize',
    '/content/llama.cpp/build/llama-quantize',
]
QBIN = next((p for p in candidates if os.path.exists(p)), None)
assert QBIN, 'llama-quantize binary not found — Step 6 failed'
print('Using:', QBIN)

subprocess.check_call([QBIN, F16_GGUF, Q_GGUF, QUANT])

print(f'\nQuantised GGUF: {Q_GGUF}  ({os.path.getsize(Q_GGUF)/1e9:.2f} GB)')

In [ ]:
import re
from llama_cpp import Llama

llm = Llama(
    model_path=Q_GGUF,
    n_ctx=2048,
    n_gpu_layers=-1,   # offload to GPU
    chat_format='gemma',
    verbose=False,
)

SYSTEM_PROMPT = (
    'You are Saheli, a maternal health assistant for ASHA workers. '
    'Given the patient symptoms and vitals, output a JSON tool call to '
    'assess_danger_signs followed by a short RED/YELLOW/GREEN recommendation '
    'in plain language. Use WHO Antenatal Care guidelines.'
)

CASES = [
    ('Patient is 32 weeks pregnant and has severe headache and blurred vision. BP 150/100.', 'RED'),
    ('28-week patient, baby has not moved all day.',                                         'RED'),
    ('36-week patient, mild ankle swelling. BP 130/85.',                                     'YELLOW'),
    ('ASHA visit: 24w, feeling very weak, haemoglobin 6.5.',                                 'YELLOW'),
    ('16-week patient, some nausea in the mornings.',                                        'GREEN'),
]

correct = 0
for user_msg, expected in CASES:
    resp = llm.create_chat_completion(
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': user_msg},
        ],
        max_tokens=200,
        temperature=0.1,
    )
    out = resp['choices'][0]['message']['content']
    m = re.search(r'\b(RED|YELLOW|GREEN)\b', out.upper())
    pred = m.group(1) if m else '?'
    ok = pred == expected
    correct += int(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] expected {expected:<6} got {pred:<6} | {user_msg[:70]}")
    print(f'        output: {out[:150]}')

print(f'\nSanity: {correct}/{len(CASES)} correct')

In [ ]:
if PUSH_GGUF_TO_HF:
    from huggingface_hub import upload_file
    upload_file(
        path_or_fileobj = Q_GGUF,
        path_in_repo    = os.path.basename(Q_GGUF),
        repo_id         = ADAPTER_REPO,
        repo_type       = 'model',
    )
    print(f'Pushed → https://huggingface.co/{ADAPTER_REPO}/blob/main/{os.path.basename(Q_GGUF)}')
else:
    print('PUSH_GGUF_TO_HF=False — skip.')

## Step 11 — Download GGUF to your computer

Run this cell to pull the `.gguf` file directly. Save it into your local project at `saheli/models/`.

In [ ]:
from google.colab import files
files.download(Q_GGUF)

## Next steps (local machine)

1. Move the downloaded `.gguf` to `saheli/models/gemma-4-e4b-q4_k_m.gguf`.
2. Update `config/settings.py`:
   ```python
   MODEL_PATH = './models/gemma-4-e4b-q4_k_m.gguf'
   ```
3. Run the three-row benchmark:
   ```bash
   python finetune/evaluate_model.py
   ```
4. Copy the Rule-only / Pure LLM / LLM+Rule numbers into the Kaggle writeup.